# RTFM + FFTFormer — Ablation Study

Runs multiple mod combinations end-to-end and plots ROC / PR curves for comparison.

**Designed to run on Google Colab.** Works on local Jupyter too — skip the Drive mount cell.

---
**Ablation order (from INSTRUCTIONS.md)**

| Step | Mods | Expected behaviour |
|------|------|-------------------|
| 0 | None (Baseline) | AUC ≈ 97.21 % on ShanghaiTech |
| 1 | 2 | AUC maintained ±0.2 % — confirms drop-in |
| 2 | 1 + 2 | AUC improves |
| 3 | 1 + 2 + 4 | Further improvement |
| 4 | 1 + 2 + 3 + 4 | Best case; revert Mod 3 if AUC drops |

## 1 · Clone the repository

In [ ]:
GITHUB_REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO_NAME'  # ← fill in before running

import os

repo_name = GITHUB_REPO_URL.rstrip('/').split('/')[-1]

if not os.path.isdir(repo_name):
    os.system(f'git clone {GITHUB_REPO_URL}')
else:
    print(f'Repository folder "{repo_name}" already exists — pulling latest changes.')
    os.system(f'git -C {repo_name} pull')

%cd {repo_name}
print('Working directory:', os.getcwd())

## 2 · Install dependencies

In [ ]:
%pip install -q torch torchvision scikit-learn matplotlib tqdm numpy

## 3 · (Colab only) Mount Google Drive
Skip this cell if running locally or if your data is already accessible.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted at /content/drive')
except ImportError:
    print('Not running in Colab — skipping Drive mount.')

## 4 · Imports

In [ ]:
import sys, types
import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

sys.path.insert(0, os.path.abspath('.'))

from model import Model
from dataset import Dataset
from train import train
from test_10crop import test
from config import Config
from utils import Visualizer, save_best_record

print('All imports OK')
print('Device:', 'GPU —', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 5 · Configuration
**Edit this cell** to point at your feature files and set the training budget.

In [ ]:
# ── Dataset / feature paths ───────────────────────────────────────────────────
# If your .npy feature files live on Google Drive, use paths like:
#   '/content/drive/MyDrive/RTFM_data/ucf-i3d.list'
RGB_LIST      = 'list/ucf-i3d.list'
TEST_RGB_LIST = 'list/ucf-i3d-test.list'
GT_FILE       = 'list/gt-ucf-local.npy'
DATASET       = 'ucf'       # 'ucf' | 'shanghai'
FEATURE_SIZE  = 2048        # 2048 for I3D, 4096 for C3D

# ── Training budget ───────────────────────────────────────────────────────────
MAX_EPOCH  = 100            # use 15000 for full RTFM training
BATCH_SIZE = 32
LR_SCHED   = '[0.001]*100'  # must have at least MAX_EPOCH values

# ── Ablation combinations ─────────────────────────────────────────────────────
# Each tuple: (display label, set of mod IDs to activate)
EXPERIMENTS = [
    ('Baseline',     set()),
    ('Mod 2',        {2}),
    ('Mod 1+2',      {1, 2}),
    ('Mod 1+2+4',    {1, 2, 4}),
    ('Mod 1+2+3+4',  {1, 2, 3, 4}),
]

CKPT_DIR    = './ckpt'
RESULTS_DIR = './results'
os.makedirs(CKPT_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print('Configuration set.')
print(f'  Dataset      : {DATASET}')
print(f'  Feature size : {FEATURE_SIZE}')
print(f'  Max epochs   : {MAX_EPOCH}')
print(f'  Experiments  : {[e[0] for e in EXPERIMENTS]}')

## 6 · Helpers

In [ ]:
def make_args(model_name):
    """Build an args namespace that the repo modules expect."""
    return types.SimpleNamespace(
        rgb_list        = RGB_LIST,
        test_rgb_list   = TEST_RGB_LIST,
        gt              = GT_FILE,
        dataset         = DATASET,
        feature_size    = FEATURE_SIZE,
        batch_size      = BATCH_SIZE,
        max_epoch       = MAX_EPOCH,
        lr              = LR_SCHED,
        modality        = 'RGB',
        model_name      = model_name,
        num_classes     = 1,
        gpus            = 1,
        workers         = 0,
        pretrained_ckpt = None,
        plot_freq       = 10,
    )


def mod_tag(active_mods):
    return ('mods_' + ''.join(str(m) for m in sorted(active_mods))
            if active_mods else 'baseline')


def run_experiment(label, active_mods):
    """
    Train RTFM with the given mod set for MAX_EPOCH steps.

    Returns a dict:
        label       – display name
        auc_history – list of (step, auc) for every evaluation
        best_auc    – highest AUC seen
        fpr, tpr, precision, recall – arrays from the best-AUC checkpoint
    """
    tag        = mod_tag(active_mods)
    name       = f'rtfm_{DATASET}_{tag}_'
    args       = make_args(name)
    config     = Config(args)
    device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    viz        = Visualizer()

    print(f'\n{"═"*60}')
    print(f'  {label}   |   mods: {sorted(active_mods) if active_mods else "none"}   |   {device}')
    print(f'{"═"*60}')

    # Data
    train_nloader = DataLoader(
        Dataset(args, test_mode=False, is_normal=True),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=0, pin_memory=False, drop_last=True)
    train_aloader = DataLoader(
        Dataset(args, test_mode=False, is_normal=False),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=0, pin_memory=False, drop_last=True)
    test_loader = DataLoader(
        Dataset(args, test_mode=True),
        batch_size=1, shuffle=False,
        num_workers=0, pin_memory=False)

    # Model
    model = Model(FEATURE_SIZE, BATCH_SIZE, active_mods=active_mods).to(device)

    # Optimizer — W_quant gets zero weight decay
    w_quant_params = [p for n, p in model.named_parameters() if 'W_quant' in n]
    other_params   = [p for n, p in model.named_parameters() if 'W_quant' not in n]
    if w_quant_params:
        optimizer = optim.Adam([
            {'params': other_params,   'lr': config.lr[0], 'weight_decay': 0.005},
            {'params': w_quant_params, 'lr': config.lr[0] * 2, 'weight_decay': 0.0},
        ])
    else:
        optimizer = optim.Adam(model.parameters(),
                               lr=config.lr[0], weight_decay=0.005)

    # State
    auc_history  = []
    best_auc     = -1
    best_curves  = {}

    # Initial eval (step 0, untrained)
    auc0, *_ = test(test_loader, model, args, viz, device)
    auc_history.append((0, auc0))

    for step in tqdm(range(1, MAX_EPOCH + 1), desc=label):
        # LR schedule
        if step > 1 and config.lr[step - 1] != config.lr[step - 2]:
            for pg in optimizer.param_groups:
                pg['lr'] = config.lr[step - 1]

        if (step - 1) % len(train_nloader) == 0:
            loadern_iter = iter(train_nloader)
        if (step - 1) % len(train_aloader) == 0:
            loadera_iter = iter(train_aloader)

        train(loadern_iter, loadera_iter, model, BATCH_SIZE, optimizer, viz, device)

        auc, fpr, tpr, precision, recall = test(test_loader, model, args, viz, device)
        auc_history.append((step, auc))

        if auc > best_auc:
            best_auc    = auc
            best_curves = dict(fpr=fpr, tpr=tpr, precision=precision, recall=recall)
            torch.save(model.state_dict(),
                       os.path.join(CKPT_DIR, name + 'best-i3d.pkl'))
            for key, arr in best_curves.items():
                np.save(os.path.join(RESULTS_DIR, f'{name}{key}.npy'), arr)
            save_best_record(
                {'epoch': [step], 'test_AUC': [auc]},
                os.path.join(RESULTS_DIR, name + 'best-AUC.txt'))

    torch.save(model.state_dict(), os.path.join(CKPT_DIR, name + 'final.pkl'))
    print(f'  ✓ Best AUC: {best_auc:.4f}')

    return dict(label=label, auc_history=auc_history,
                best_auc=best_auc, **best_curves)

## 7 · Run all experiments
Results accumulate in `all_results`. Each entry is a dict returned by `run_experiment`.

In [ ]:
all_results = []

for label, active_mods in EXPERIMENTS:
    result = run_experiment(label, active_mods)
    all_results.append(result)

print('\n' + '='*40)
print(f'{"Configuration":<20}  {"Best AUC"}')
print('-'*40)
for r in all_results:
    print(f'{r["label"]:<20}  {r["best_auc"]:.4f}')

## 8 · Plot: AUC over training steps

In [ ]:
colors = cm.tab10(np.linspace(0, 1, len(all_results)))

fig, ax = plt.subplots(figsize=(11, 5))

for r, c in zip(all_results, colors):
    steps, aucs = zip(*r['auc_history'])
    ax.plot(steps, aucs,
            label=f"{r['label']}  (best={r['best_auc']:.4f})",
            color=c, linewidth=1.8)

ax.set_xlabel('Training step', fontsize=12)
ax.set_ylabel('AUC', fontsize=12)
ax.set_title('AUC over Training Steps — Ablation', fontsize=13)
ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'auc_training_curves.png'), dpi=150)
plt.show()

## 9 · Plot: ROC curves (best checkpoint per experiment)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random')

for r, c in zip(all_results, colors):
    if 'fpr' not in r:
        continue
    ax.plot(r['fpr'], r['tpr'],
            label=f"{r['label']}  (AUC={r['best_auc']:.4f})",
            color=c, linewidth=1.8)

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Best Model per Configuration', fontsize=13)
ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'roc_curves.png'), dpi=150)
plt.show()

## 10 · Plot: Precision-Recall curves (best checkpoint per experiment)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for r, c in zip(all_results, colors):
    if 'precision' not in r:
        continue
    order  = np.argsort(r['recall'])
    ap     = float(np.trapz(r['precision'][order], r['recall'][order]))
    ax.plot(r['recall'][order], r['precision'][order],
            label=f"{r['label']}  (AP={ap:.4f})",
            color=c, linewidth=1.8)

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves — Best Model per Configuration', fontsize=13)
ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'pr_curves.png'), dpi=150)
plt.show()

## 11 · Plot: Best AUC bar chart

In [ ]:
labels    = [r['label']    for r in all_results]
best_aucs = [r['best_auc'] for r in all_results]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, best_aucs, color=colors, edgecolor='black', linewidth=0.6)

for bar, val in zip(bars, best_aucs):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.0005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)

ax.set_ylim(max(0, min(best_aucs) - 0.02), min(1.0, max(best_aucs) + 0.02))
ax.set_ylabel('Best AUC', fontsize=12)
ax.set_title('Best AUC per Configuration', fontsize=13)
ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'best_auc_bar.png'), dpi=150)
plt.show()

## 12 · (Optional) Re-plot from saved `.npy` files
Run this section independently to regenerate all plots after training has finished,
without re-running the training loop.

In [ ]:
def load_results(experiments, results_dir=RESULTS_DIR):
    """Reload best-curve .npy files written during training."""
    loaded = []
    for label, active_mods in experiments:
        tag    = mod_tag(active_mods)
        name   = f'rtfm_{DATASET}_{tag}_'
        prefix = os.path.join(results_dir, name)
        try:
            auc_line = open(prefix + 'best-AUC.txt').readlines()[-1].strip()
            best_auc = float(auc_line)
            loaded.append(dict(
                label     = label,
                best_auc  = best_auc,
                fpr       = np.load(prefix + 'fpr.npy'),
                tpr       = np.load(prefix + 'tpr.npy'),
                precision = np.load(prefix + 'precision.npy'),
                recall    = np.load(prefix + 'recall.npy'),
            ))
            print(f'  Loaded : {label:<20}  AUC={best_auc:.4f}')
        except FileNotFoundError:
            print(f'  Skipped (no saved results): {label}')
    return loaded


# Uncomment the two lines below to reload and re-plot without retraining:
# all_results = load_results(EXPERIMENTS)
# # Then re-run cells 8–11 above.